# 04 · Mapa de temas

Duas perguntas do enunciado pedem frequência: "quais problemas aparecem com maior frequência" e "quais reclamações de entrega aparecem com maior frequência". Um top-10 de avaliações recuperadas não sustenta uma afirmação de frequência sobre 42 mil textos. **D10:** a frequência sai de um mapa de temas da base inteira, contado em código; o LLM só redige com os números que recebe.

O mapa usa BERTopic (aula de NLP 6) sobre os embeddings já calculados no notebook 03, e é comparado com LDA pela coerência c_v. O código está em `src/indexing/temas.py`; `make index` grava o tema de cada texto em `data/index/temas.parquet` e o resumo em `reports/temas.csv`.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import logging
import time
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
os.environ["TQDM_DISABLE"] = "1"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.indexing.temas import SIDES, fit_topics, side_of, topic_words
from src.visualization.estilo import BLUE, GRAY, GREEN, ORANGE, RED, apply_style, fmt_pct, save

apply_style()
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)
docs = pd.read_parquet("data/processed/documentos.parquet")

## 1. Um mapa por polaridade

O notebook 03 mostrou que o embedding pesa pouco a polaridade. Num teste com um modelo único sobre toda a base, os tópicos de entrega juntavam "não foi entregue" e "chegou 15 dias antes do prazo". O gestor pergunta por problemas ou por elogios, nunca pelos dois misturados. **D23:** dois modelos, um sobre os textos com nota média até 3 (problemas) e outro sobre os textos com nota média acima de 3 (elogios). A nota 3 fica do lado dos problemas, e a seção 4 confirma a escolha: mais de 80% das avaliações de nota 3 caem em temas de problema, e menos de 17% em "satisfeito, com ressalvas".

A nota média é calculada por texto distinto (o mesmo texto pode ter avaliações com notas diferentes, D04); só 9,2% dos textos ficam estritamente entre 2 e 4, então a divisão quase não depende do ponto de corte.

In [2]:
media = docs.groupby("doc_id")["review_score"].mean()
lado = pd.Series(side_of(media.to_numpy()), index=media.index)
resumo = pd.DataFrame({
    "textos distintos": lado.value_counts(),
    "avaliações": docs["doc_id"].map(lado).value_counts(),
})
print(f"textos com nota média estritamente entre 2 e 4: {100 * media.between(2, 4, inclusive='neither').mean():.1f}%")
resumo

textos com nota média estritamente entre 2 e 4: 9.2%


,textos distintos,avaliações
elogio,21471,28070
problema,13418,14023


## 2. BERTopic

Configuração de cada lado: UMAP para 5 dimensões com distância de cosseno, HDBSCAN (grupos de pelo menos 40 textos no lado de problemas e 100 no de elogios, que é maior e mais repetitivo), representação dos tópicos por c-TF-IDF com as stopwords do projeto, redução hierárquica para 20 e 15 tópicos e realocação dos textos que o HDBSCAN deixou sem tópico (32% a 39%) pelo embedding mais próximo, com limiar de similaridade de 0,5. Os parâmetros saíram de testes sobre a base inteira com 50, 100, 200 e 400 como tamanho mínimo: com 50 e 100 sobravam 110 e 64 tópicos, muitos redundantes; com 200, um único tópico ficava com 48% dos textos. Por isso o tamanho mínimo é moderado e o número final de tópicos vem da redução hierárquica.

In [3]:
t0 = time.perf_counter()
assign, models = fit_topics(docs)
print(f"ajuste dos dois modelos em {time.perf_counter() - t0:.0f} s")
for side, model in models.items():
    info = model.get_topic_info()
    print(f"{side}: {len(info[info.Topic >= 0])} tópicos, {SIDES[side]}")

ajuste dos dois modelos em 93 s
problema: 19 tópicos, {'min_cluster_size': 40, 'nr_topics': 20}
elogio: 14 tópicos, {'min_cluster_size': 100, 'nr_topics': 15}


In [4]:
rev = docs.merge(assign, on="doc_id")
def tabela_topicos(side):
    r = rev[rev["polaridade"] == side]
    g = r.groupby("topico").agg(avaliacoes=("review_id", "size"), nota=("review_score", "mean")).round(2)
    g["palavras"] = [", ".join(topic_words(models[side], t, 7)) for t in g.index]
    g["exemplo"] = [r[r.topico == t].drop_duplicates("doc_id").sample(1, random_state=3)["text_clean"].iloc[0][:70] for t in g.index]
    return g.sort_values("avaliacoes", ascending=False)
tabela_topicos("problema")

,avaliacoes,nota,palavras,exemplo
topico,,,,
0,4143,1.35,"nao, produto, nao recebi, recebi, ainda, recebi produto, ainda nao","sempre comprei na lannister , mas desta vez o perceiro deles nao entre"
2,2081,1.65,"produto, nao, qualidade, veio, produto nao, produto veio, defeito",Não me entregaram os produtos
1,1532,1.52,"comprei, dois, apenas, recebi, produtos, unidades, duas",Comprei 3 itens. recebi apenas 2. e a loja não responde meus contatos.
7,875,1.86,"prazo, entrega, dias, demora, demorou, prazo entrega, longo",Passou do prazo de e trega
3,787,2.52,"bom, produto, produto bom, recomendo, gostei, entrega, prazo","gostei da torneira, veio com uma oxidação na lateral.uns respingos. ma"
4,764,1.75,"cor, foto, diferente, produto, veio, nao, veio cor","Produto comercializado não é o original da marca Oakley, trata-se de r"
5,582,1.69,"celular, nao, pelicula, veio, funciona, cabo, cartucho","Pedi HD 500 GB SATA Notebook Samsung R520 1 unidade - R$ 269,00 cada E"
8,507,1.52,"nao, nao recebi, recebi, ainda, ainda nao, nada, avaliar",nao era o que eu esperava.achei o relogio pequena
6,471,1.53,"nao, produto, recebi, nao recebi, entrega, compra, ainda","COMPREI DOIS E RECEBI APENAS 1, SEGUNDO O ATENDENTE A SEGUNDA ESTAR PA"


In [5]:
tabela_topicos("elogio")

,avaliacoes,nota,palavras,exemplo
topico,,,,
0,7500,4.79,"produto, recomendo, prazo, loja, qualidade, bom, entregue",capa de notbuk. Veio conforme o anunciado simples mais bonita.
2,5979,4.76,"tudo, bom, recomendo, otimo, gostei, super, adorei",Produto excelente. Cadiveu é maravilhoso!
1,5391,4.80,"antes, prazo, antes prazo, chegou, chegou antes, produto, bem","Aprovado. Produto entregue antes do prazo, + n considero fenômeno, já"
3,2177,4.85,"rapida, rapido, entrega, entrega rapida, super, rapida produto, entrega super",Rápido demais chegou ja usamos é bom o produto parabéns
6,1988,4.63,"prazo, produto, recomendo, bom, antes, otimo, entregue",BOA. AINDA NÃO INSTALEI AINDA.MAIS TIVE EXCELENTE INDICAÇÃO
4,1374,4.44,"nao, ainda, ainda nao, produto, veio, porem, bom",Excelente serviço. Falta apenas mais instruções como usar os produtos.
5,911,4.51,"nao, correios, recebi, comprei, correio, produto, pedido","Atenciosos, culpa do atraso foi dos correios."
-1,767,4.57,"nao, nunca, sempre, frete, problemas, veio, sem",Comprei na quinta e chegou na segunda
7,581,4.86,"atendimento, empresa, cliente, servico, clientes, parabens, excelente",Empresa comprometida com o atendimento ao cliente e idônea nos produto


A comparação com LDA e os temas nomeados vêm nas próximas etapas deste notebook.